# Preprocessing the dataset
1. The raw dataset being processed is stored at location: /datasets/Raw-Time-Series-Data.csv
2. The output dataset is saved at location: /datasets/Cleaned-Time-Series-Data.csv

## Imports

In [68]:
import os
import pandas as pd

## Loading CSV file

In [69]:
csv_file_path = '../datasets/Raw-Time-Series-Data.csv'

try:
    df = pd.read_csv(csv_file_path)
    print(f"Successfully loaded {csv_file_path}")
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found.")
except Exception as e:
    print(f"An error occurred while reading the CSV file: {e}")

df

Successfully loaded ../datasets/Raw-Time-Series-Data.csv


,system:index,BRGY_NAME,bare_ground,built_area,crops,flooded_vegetation,grass,quarter,shrub_and_scrub,snow_and_ice,trees,water,year,.geo
0,00000000000000000000,PMA FORT DEL PILAR,0.001683,17.320226,0.000000,0.0,0.035432,4,0.000842,0.000000,82.641817,0.000000,2023,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,00000000000000000001,KIAS,0.000000,51.671354,0.000000,0.0,0.751134,4,0.000000,0.000000,47.577512,0.000000,2023,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,00000000000000000002,STO. TOMAS SCHOOL AREA,0.000000,27.067083,0.755131,0.0,7.896787,4,0.830630,0.000000,63.450370,0.000000,2023,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,00000000000000000003,ATOK TRAIL,0.000000,20.784876,0.000000,0.0,0.363599,4,0.000000,0.000000,78.851525,0.000000,2023,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,00000000000000000004,LOAKAN APUGAN,0.000000,61.954822,0.000000,0.0,0.000000,4,0.000000,0.000000,38.045178,0.000000,2023,"{""type"":""MultiPoint"",""coordinates"":[]}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5155,0000000000000000007c,WEST QUIRINO HILL,0.000000,100.000000,0.000000,0.0,0.000000,3,0.000000,0.000000,0.000000,0.000000,2022,"{""type"":""MultiPoint"",""coordinates"":[]}"
5156,0000000000000000007d,PINGET,0.000000,85.245253,0.000000,0.0,0.000000,3,0.000000,0.000000,14.754747,0.000000,2022,"{""type"":""MultiPoint"",""coordinates"":[]}"
5157,0000000000000000007e,PINSAO PILOT PROJ,0.000000,99.996067,0.000000,0.0,0.000000,3,0.000000,0.000000,0.003933,0.000000,2022,"{""type"":""MultiPoint"",""coordinates"":[]}"
5158,0000000000000000007f,PINSAO PROPER,0.000000,49.420578,0.063114,0.0,0.339009,3,0.003945,0.406301,49.767053,0.000000,2022,"{""type"":""MultiPoint"",""coordinates"":[]}"


## Feature Engineering
- changing the name of 'BRGY_NAME' to 'baranggay_id'
- Removing the columns 'system:index', 'BRGY_NAME', '.geo'

In [70]:
df['baranggay_id'] = df['BRGY_NAME'].astype('category').cat.codes + 1
drop_columns = ['system:index', 'BRGY_NAME', '.geo']
df = df.drop(columns=drop_columns)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5160 entries, 0 to 5159
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bare_ground         5160 non-null   float64
 1   built_area          5160 non-null   float64
 2   crops               5160 non-null   float64
 3   flooded_vegetation  5160 non-null   float64
 4   grass               5160 non-null   float64
 5   quarter             5160 non-null   int64  
 6   shrub_and_scrub     5160 non-null   float64
 7   snow_and_ice        5160 non-null   float64
 8   trees               5160 non-null   float64
 9   water               5160 non-null   float64
 10  year                5160 non-null   int64  
 11  baranggay_id        5160 non-null   int16  
dtypes: float64(9), int16(1), int64(2)
memory usage: 453.6 KB


In [71]:
df = df.sort_values(by=['baranggay_id','year', 'quarter'])
df

,bare_ground,built_area,crops,flooded_vegetation,grass,quarter,shrub_and_scrub,snow_and_ice,trees,water,year,baranggay_id
4985,15.539500,74.658088,0.0,0.0,0.0,1,5.543881,4.116581,0.0,0.14195,2016,1
599,7.052464,83.866569,0.0,0.0,0.0,2,4.258527,4.680490,0.0,0.14195,2016,1
2921,0.000000,100.000000,0.0,0.0,0.0,3,0.000000,0.000000,0.0,0.00000,2016,1
341,1.419513,93.612204,0.0,0.0,0.0,4,0.141951,4.826332,0.0,0.00000,2016,1
4598,9.226831,76.436116,0.0,0.0,0.0,1,0.283902,14.053150,0.0,0.00000,2017,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1027,0.000000,100.000000,0.0,0.0,0.0,4,0.000000,0.000000,0.0,0.00000,2024,129
2446,0.000000,100.000000,0.0,0.0,0.0,1,0.000000,0.000000,0.0,0.00000,2025,129
2833,0.000000,100.000000,0.0,0.0,0.0,2,0.000000,0.000000,0.0,0.00000,2025,129
4123,0.000000,100.000000,0.0,0.0,0.0,3,0.000000,0.000000,0.0,0.00000,2025,129


## Normalization
- percentage values of LULC labels are normalized from 0 to 1.

In [72]:
# list of LULC columns
lulc_cols = [
    "bare_ground",
    "built_area",
    "crops",
    "flooded_vegetation",
    "grass",
    "shrub_and_scrub",
    "snow_and_ice",
    "trees",
    "water"
]

df[lulc_cols] = df[lulc_cols] / 100
df


,bare_ground,built_area,crops,flooded_vegetation,grass,quarter,shrub_and_scrub,snow_and_ice,trees,water,year,baranggay_id
4985,0.155395,0.746581,0.0,0.0,0.0,1,0.055439,0.041166,0.0,0.00142,2016,1
599,0.070525,0.838666,0.0,0.0,0.0,2,0.042585,0.046805,0.0,0.00142,2016,1
2921,0.000000,1.000000,0.0,0.0,0.0,3,0.000000,0.000000,0.0,0.00000,2016,1
341,0.014195,0.936122,0.0,0.0,0.0,4,0.001420,0.048263,0.0,0.00000,2016,1
4598,0.092268,0.764361,0.0,0.0,0.0,1,0.002839,0.140531,0.0,0.00000,2017,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1027,0.000000,1.000000,0.0,0.0,0.0,4,0.000000,0.000000,0.0,0.00000,2024,129
2446,0.000000,1.000000,0.0,0.0,0.0,1,0.000000,0.000000,0.0,0.00000,2025,129
2833,0.000000,1.000000,0.0,0.0,0.0,2,0.000000,0.000000,0.0,0.00000,2025,129
4123,0.000000,1.000000,0.0,0.0,0.0,3,0.000000,0.000000,0.0,0.00000,2025,129


## Exporting
- The file will be exported to location: /dataseta/Clean-Time-Series-Data.csv

In [73]:
# 1. Define your path
csv_output_path = '../datasets/Clean-Time-Series-Data.csv'

# 2. Ensure the directory exists (optional but recommended)
os.makedirs(os.path.dirname(csv_output_path), exist_ok=True)

# 3. Export the DataFrame
# index=False prevents pandas from writing the row numbers as a separate column
df.to_csv(csv_output_path, index=False)

print(f"Dataset successfully saved to: {csv_output_path}")

Dataset successfully saved to: ../datasets/Clean-Time-Series-Data-1.csv
